# Normalización del dataset de poblacion

### 1. Importar librerias necesarias

In [1]:
import pandas as pd
import numpy as np
import unicodedata
import os

from pathlib import Path
from difflib import get_close_matches

### 2. Carga del dataset a normalizar

In [3]:
# Definir la ruta del dataset
dataset_path = r'C:\00 - Proyecto Incendios Galicia 8.0\data\01 - Originales\04 - Poblacion\04 - Poblacion.parquet'

# Cargar el archivo CSV directamente
df = pd.read_parquet(dataset_path)

print(f'Filas cargadas: {len(df)}')
print('Columnas disponibles:', list(df.columns))
print(f'Tamaño del dataset: {df.shape[0]} filas x {df.shape[1]} columnas')
display(df.head())

# Mostrar el número de municipios únicos en la columna 'municipio'
print(f"Municipios únicos en el dataset: {df['municipio'].nunique()}")

Filas cargadas: 2921920
Columnas disponibles: ['POP', 'pop_density', 'pop_growth', 'area_km2', 'fecha', 'municipio', 'latitud', 'longitud']
Tamaño del dataset: 2921920 filas x 8 columnas


,POP,pop_density,pop_growth,area_km2,fecha,municipio,latitud,longitud
0,8.048130,0.097686,-0.000091,82.388065,1995-01-01,ferrol,43.484571,-8.232997
1,1.516606,0.052624,-0.000015,28.819931,1995-01-01,fisterra,42.906477,-9.263789
2,0.617464,0.015417,-0.000006,40.050962,1995-01-01,paderne,43.296179,-8.156219
3,1.754705,0.035964,-0.000024,48.790271,1995-01-01,padron,42.739023,-8.660250
4,0.345283,0.002651,-0.000003,130.252600,1995-01-01,o pino,42.945490,-8.350221


Municipios únicos en el dataset: 317


In [4]:
# Mostrar la primera fila como un diccionario columna: valor
primera_fila_dict = df.iloc[0].to_dict()
for k, v in primera_fila_dict.items():
    print(f"{k} = {v}")

POP = 8.04812974820207
pop_density = 0.0976856259817802
pop_growth = -9.102190812193222e-05
area_km2 = 82.38806546321526
fecha = 1995-01-01 00:00:00
municipio = ferrol
latitud = 43.4845713
longitud = -8.2329968


### 2.1 Normalizar los nombres de las columnas

In [6]:
# Normalizar nombres de columnas a español, minúsculas y descriptivos según el dataset actual
columnas_renombrar = {
    'POP': 'poblacion',
    'pop_density': 'densidad_poblacion',
    'pop_growth': 'crecimiento_poblacion',
    'area_km2': 'superficie_km2',
    'fecha': 'fecha',
    'municipio': 'municipio',
    'latitud': 'latitud',
    'longitud': 'longitud',
}
df.rename(columns=columnas_renombrar, inplace=True)
df.columns = [col.lower() for col in df.columns]
print('Nombres de columnas tras la normalización:')
print(list(df.columns))

Nombres de columnas tras la normalización:
['poblacion', 'densidad_poblacion', 'crecimiento_poblacion', 'superficie_km2', 'fecha', 'municipio', 'latitud', 'longitud']


### 3. Visualización y exploración inicial

In [7]:
# Ver las primeras filas del dataset
df.head()

# Ver información general del dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2921920 entries, 0 to 2921919
Data columns (total 8 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   poblacion              float64       
 1   densidad_poblacion     float64       
 2   crecimiento_poblacion  float64       
 3   superficie_km2         float64       
 4   fecha                  datetime64[ns]
 5   municipio              object        
 6   latitud                float64       
 7   longitud               float64       
dtypes: datetime64[ns](1), float64(6), object(1)
memory usage: 178.3+ MB


### 4. Cargar el dataset limpio de municipios de Galicia

In [8]:
# Ruta del archivo de municipios limpios
municipios_path = r'C:\00 - Proyecto Incendios Galicia 8.0\data\02 - Municipio normalizado\01 - municipios\01 - Tabla de municipios.csv'

# Cargar el archivo Excel de municipios
df_municipios = pd.read_csv(municipios_path)

# Mostrar las primeras filas para comprobar que se ha cargado bien
display(df_municipios.head())

# Mostrar el número de municipios únicos en el dataset de municipios
print(f"Municipios únicos en el dataset de municipios: {df_municipios['municipio'].nunique()}")

,municipio,comarca,provincia,altitud,superficie,poblacion,densidad
0,a arnoia,comarca del ribeiro,ourense,76,"20,69",1.000,"48,33"
1,a baña,barcala,a coruña,297,"98,19",3.450,"35,14"
2,a bola,comarca de tierra de celanova,ourense,510,"34,9",1.156,"33,12"
3,a capela,comarca del eume,a coruña,NaN,58,1.232,"21,24"
4,a cañiza,comarca de paradanta,pontevedra,570,"105,04",5.180,"49,31"


Municipios únicos en el dataset de municipios: 315


In [9]:
# Comparar nombres de municipio entre ambos datasets (df y df_municipios)
municipios_dataset = set(df['municipio'].dropna().unique())
municipios_referencia = set(df_municipios['municipio'].dropna().unique())

# Municipios que están en ambos datasets
municipios_comunes = municipios_dataset & municipios_referencia

# Municipios que están en el dataset principal pero no en la referencia
municipios_solo_dataset = municipios_dataset - municipios_referencia

# Municipios que están en la referencia pero no en el dataset principal
municipios_solo_referencia = municipios_referencia - municipios_dataset

print(f"Total municipios en dataset principal: {len(municipios_dataset)}")
print(f"Total municipios en referencia: {len(municipios_referencia)}")
print(f"Municipios coincidentes: {len(municipios_comunes)}")
print(f"Municipios solo en dataset principal: {municipios_solo_dataset}")
print(f"Municipios solo en referencia: {municipios_solo_referencia}")

Total municipios en dataset principal: 317
Total municipios en referencia: 315
Municipios coincidentes: 230
Municipios solo en dataset principal: {'mondonedo', 'morana', 'meano', 'padron', 'lalin', 'cabana de bergantinos', 'muxia', 'guntin', 'becerrea', 'melon', 'a gudina', 'a bana', 'porqueira', 'cesuras', 'marin', 'naron', 'xunqueira de ambia', 'oimbra', 'sandias', 'san xoan de rio', 'cerdedo cotobade', 'portomarin', 'punxin', 'lourenza', 'nigran', 'mondariz balneario', 'melide', 'verin', 'coiros', 'rubia', 'o carballino', 'dozon', 'pazos de borben', 'vilamartin de valdeorras', 'avion', 'toen', 'mino', 'camarinas', 'o porrino', 'oza cesuras', 'dumbria', 'san sadurnino', 'a rua', 'abadin', 'o savinao', 'a caniza', 'salvaterra de mino', 'a coruna', 'muinos', 'arzua', 'o paramo', 'manon', 'brion', 'boveda', 'lobeira', 'vilamarin', 'nogueira de ramuin', 'corcubion', 'san cibrao das vinas', 'as pontes de garcia rodriguez', 'pinor', 'ramiras', 'negueira de muniz', 'calvos de randin', 'cast

### 5. Normalización automática de municipios

In [ ]:
# Normalización directa de municipios en la columna 'municipio' (sin duplicar filas ni columnas)

col_municipio = 'municipio'
col_ref = 'municipio'
df_ref = df_municipios

# Función para normalizar nombres eliminando tildes, mayúsculas, signos y espacios extra
def normalizar_nombre(nombre):
    if pd.isnull(nombre):
        return ''
    nombre = str(nombre).strip().lower()
    nombre = ''.join(c for c in unicodedata.normalize('NFD', nombre) if unicodedata.category(c) != 'Mn')
    nombre = nombre.replace('-', ' ').replace(',', '').replace('.', '')
    nombre = ' '.join(nombre.split())
    return nombre

# Diccionario de referencia normalizada para búsqueda rápida
ref_norm = {normalizar_nombre(x): x for x in df_ref[col_ref].dropna().unique()}
ref_norm_keys = set(ref_norm.keys())

# 1. Obtener todos los valores únicos del dataset a normalizar y de la referencia
df[col_municipio] = df[col_municipio].astype(str)
municipios_unicos = set(x for x in df[col_municipio].unique() if isinstance(x, str) and x.strip())
municipios_referencia = set(df_ref[col_ref].dropna().unique())

# 2. Crear mapeo: municipio original -> municipio normalizado (o sugerido, o pendiente)
mapeo = {}
pendientes = []
for m in municipios_unicos:
    clave = normalizar_nombre(m)
    if clave in ref_norm:
        mapeo[m] = ref_norm[clave]
        continue
    sugerencias = get_close_matches(clave, ref_norm_keys, n=1, cutoff=0.8)
    if sugerencias:
        mapeo[m] = ref_norm[sugerencias[0]]
        continue
    partes = clave.split()
    if len(partes) == 2:
        invertido = ' '.join(partes[::-1])
        if invertido in ref_norm:
            mapeo[m] = ref_norm[invertido]
            continue
        sugerencias_inv = get_close_matches(invertido, ref_norm_keys, n=1, cutoff=0.8)
        if sugerencias_inv:
            mapeo[m] = ref_norm[sugerencias_inv[0]]
            continue
    mapeo[m] = m
    pendientes.append(m)

# 3. Aplicar el mapeo directamente sobre la columna 'municipio'
df[col_municipio] = df[col_municipio].map(mapeo)

# 4. Corrección especial: fusionar 'Oza dos Ríos' y 'Cesuras' en 'oza-cesuras'
df[col_municipio] = df[col_municipio].replace({'Oza dos Ríos': 'oza-cesuras', 'Cesuras': 'oza-cesuras'})

# 5. Diagnóstico de diferencias entre dataset y referencia
municipios_normalizados = set(df[col_municipio].dropna().unique())
faltan_en_dataset = municipios_referencia - municipios_normalizados
sobran_en_dataset = municipios_normalizados - municipios_referencia

print(f"Municipios únicos en el dataset de referencia: {len(municipios_referencia)}")
print(f"Municipios únicos normalizados en el dataset principal: {len(municipios_normalizados)}")
if faltan_en_dataset:
    print(f"Municipios de la referencia que NO aparecen en el dataset principal: {faltan_en_dataset}")
else:
    print("Todos los municipios de la referencia están presentes en el dataset principal.")
if sobran_en_dataset:
    print(f"Municipios en el dataset principal que NO están en la referencia: {sobran_en_dataset}")
else:
    print("No hay municipios extra en el dataset principal.")

# Eliminar columna 'Municipio_normalizado' si existe
if 'Municipio_normalizado' in df.columns:
    df.drop(columns=['Municipio_normalizado'], inplace=True)

Municipios únicos en el dataset de referencia: 315
Municipios únicos normalizados en el dataset principal: 317
Todos los municipios de la referencia están presentes en el dataset principal.
Municipios en el dataset principal que NO están en la referencia: {'cesuras', 'oza dos rios'}


In [11]:
# Corrección manual de nombres de municipios para casos especiales
df['municipio'] = df['municipio'].replace({'cesuras': 'oza-cesuras', 'oza dos rios': 'oza-cesuras'})
print("Corrección manual aplicada: 'cesuras' y 'oza dos rios' ahora son 'oza-cesuras'.")
print(f"Municipios únicos tras la corrección manual: {df['municipio'].nunique()}")

Corrección manual aplicada: 'cesuras' y 'oza dos rios' ahora son 'oza-cesuras'.
Municipios únicos tras la corrección manual: 315


### 6. Exportar el dataset final con municipios normalizados

In [14]:
# Exportar el dataframe final con municipios normalizados (sin columnas duplicadas)

ruta_export = r'C:\00 - Proyecto Incendios Galicia 8.0\data\02 - Municipio normalizado\04 - Poblacion'
os.makedirs(ruta_export, exist_ok=True)
archivo_export = os.path.join(ruta_export, 'poblacion galicia municipios normalizados.csv')

# Eliminar columnas duplicadas si las hubiera
df = df.loc[:, ~df.columns.duplicated()]

df.to_csv(archivo_export, index=False, encoding='utf-8')
print(f'Dataset final exportado como {archivo_export}')

Dataset final exportado como C:\00 - Proyecto Incendios Galicia 8.0\data\02 - Municipio normalizado\04 - Poblacion\poblacion galicia municipios normalizados.csv


In [15]:
# Comprobación final: número de municipios únicos en el CSV exportado vs referencia

csv_exportado = r'C:\00 - Proyecto Incendios Galicia 8.0\data\02 - Municipio normalizado\04 - Poblacion\poblacion galicia municipios normalizados.csv'
df_exportado = pd.read_csv(csv_exportado)

municipios_exportados = set(df_exportado['municipio'].dropna().unique())
print(f"Municipios únicos en el CSV exportado: {len(municipios_exportados)}")
print(f"Municipios únicos en la referencia oficial: {len(municipios_referencia)}")

if municipios_exportados == municipios_referencia:
    print('¡El CSV exportado contiene exactamente los mismos municipios que la referencia oficial!')
else:
    diferencia = municipios_exportados.symmetric_difference(municipios_referencia)
    print(f"Diferencias encontradas: {diferencia}")

Municipios únicos en el CSV exportado: 315
Municipios únicos en la referencia oficial: 315
¡El CSV exportado contiene exactamente los mismos municipios que la referencia oficial!


In [16]:
# Leer el dataset exportado y mostrar las 10 primeras filas
csv_exportado = r'C:\00 - Proyecto Incendios Galicia 7.0\data\02 - Municipio normalizado\04 - Poblacion\poblacion galicia municipios normalizados.csv'
df_exportado = pd.read_csv(csv_exportado)
display(df_exportado.head(10))

,poblacion,densidad_poblacion,crecimiento_poblacion,superficie_km2,fecha,municipio,latitud,longitud
0,8.048130,0.097686,-0.000091,82.388065,1995-01-01,ferrol,43.484571,-8.232997
1,1.516606,0.052624,-0.000015,28.819931,1995-01-01,fisterra,42.906477,-9.263789
2,0.617464,0.015417,-0.000006,40.050962,1995-01-01,paderne,43.296179,-8.156219
3,1.754705,0.035964,-0.000024,48.790271,1995-01-01,padrón,42.739023,-8.660250
4,0.345283,0.002651,-0.000003,130.252600,1995-01-01,o pino,42.945490,-8.350221
5,2.788024,0.087132,-0.000026,31.997723,1995-01-01,a pobra do caramiñal,42.604625,-8.940205
6,0.625115,0.006826,-0.000006,91.574903,1995-01-01,ponteceso,43.242818,-8.900505
7,2.663744,0.089986,-0.000026,29.601649,1995-01-01,pontedeume,43.407258,-8.171882
8,0.418156,0.001669,-0.000004,250.603548,1995-01-01,as pontes de garcía rodríguez,43.450218,-7.853109
9,0.959923,0.010098,-0.000010,95.059319,1995-01-01,porto do son,42.726023,-9.004912
